# Next Sentence Prediction with BERT

This one is a small demo — no training. BERT's pre-training has two heads: masked language modeling, and **next sentence prediction (NSP)** — given sentence A and sentence B, predict whether B actually follows A in the corpus.

The NSP head is famously not great (RoBERTa dropped it; the consensus is that MLM does most of the work). But it's still useful as a quick sanity check on what BERT considers "plausibly adjacent", and the `BertForNextSentencePrediction` class makes it easy to play with.

What I'm doing here:
1. Load `bert-base-uncased` with the NSP head.
2. Hand-craft a few sentence pairs — some natural follow-ups, some intentionally non-sequiturs.
3. Run inference and print the NSP probability for each pair.

No training, no labels, no fine-tuning. Pure inference demo. If the HF download fails, the notebook says so and exits cleanly.


In [1]:
# !pip install transformers torch -q
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np

try:
    from transformers import BertTokenizer, BertForNextSentencePrediction
    HAS_HF = True
except Exception as e:
    HAS_HF = False
    print("transformers import failed:", e)


In [2]:
if HAS_HF:
    try:
        tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
        model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased")
        model.eval()
        print("Loaded bert-base-uncased + NSP head")
    except Exception as e:
        HAS_HF = False
        print("HF model download failed (likely no internet):", e)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loaded bert-base-uncased + NSP head


## Some pairs to score

In [3]:
# (sentence_a, sentence_b, my_label_for_my_own_reference)
# label 'follows' = I expect this to be a plausible follow-up
# label 'random'  = I deliberately picked something unrelated
pairs = [
    ("The dog ran across the yard.",                 "It came back panting after a few minutes.",          "follows"),
    ("The dog ran across the yard.",                 "Quarterly earnings beat analyst expectations.",      "random"),

    ("She opened the cookbook to page 42.",          "The recipe called for two cups of flour.",            "follows"),
    ("She opened the cookbook to page 42.",          "The Eiffel Tower stands 330 metres tall.",            "random"),

    ("The package arrived on Tuesday.",              "I tore it open immediately.",                         "follows"),
    ("The package arrived on Tuesday.",              "Plants use chlorophyll to capture light.",            "random"),

    ("He drafted an email to his manager.",          "He read it through twice before hitting send.",        "follows"),
    ("He drafted an email to his manager.",          "The whale broke the surface and dove again.",         "random"),

    ("Boil the water for about three minutes.",      "Then add the noodles.",                                "follows"),
    ("Boil the water for about three minutes.",      "Beethoven composed nine symphonies.",                  "random"),
]
print(f"{len(pairs)} pairs to score")


10 pairs to score


## Run NSP inference

`BertForNextSentencePrediction` outputs logits over two classes: `IsNext` (label 0) and `NotNext` (label 1). The "probability of IsNext" is `softmax(logits)[:, 0]`. That's the number I want.


In [4]:
if HAS_HF:
    def nsp_prob_is_next(sent_a, sent_b):
        enc = tokenizer(sent_a, sent_b, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).squeeze().numpy()
        return float(probs[0])   # IsNext prob

    print(f"{'truth':<8}  {'P(IsNext)':>10}  sentence pair (truncated)")
    print("-" * 80)
    n_correct = 0
    for a, b, label in pairs:
        p = nsp_prob_is_next(a, b)
        pred = "follows" if p > 0.5 else "random"
        if pred == label:
            n_correct += 1
        marker = "ok" if pred == label else "miss"
        print(f"{label:<8}  {p:>10.3f}  [{marker}]  '{a[:35]}...' | '{b[:35]}'")
    print(f"\nNSP head agreed with my labels on {n_correct}/{len(pairs)} pairs.")
else:
    print("Cannot run inference without the HF model. Exiting cleanly.")


truth      P(IsNext)  sentence pair (truncated)
--------------------------------------------------------------------------------


follows        1.000  [ok]  'The dog ran across the yard....' | 'It came back panting after a few mi'
random         0.000  [ok]  'The dog ran across the yard....' | 'Quarterly earnings beat analyst exp'


follows        1.000  [ok]  'She opened the cookbook to page 42....' | 'The recipe called for two cups of f'
random         0.000  [ok]  'She opened the cookbook to page 42....' | 'The Eiffel Tower stands 330 metres '


follows        1.000  [ok]  'The package arrived on Tuesday....' | 'I tore it open immediately.'
random         0.000  [ok]  'The package arrived on Tuesday....' | 'Plants use chlorophyll to capture l'


follows        1.000  [ok]  'He drafted an email to his manager....' | 'He read it through twice before hit'
random         0.000  [ok]  'He drafted an email to his manager....' | 'The whale broke the surface and dov'
follows        1.000  [ok]  'Boil the water for about three minu...' | 'Then add the noodles.'


random         0.000  [ok]  'Boil the water for about three minu...' | 'Beethoven composed nine symphonies.'

NSP head agreed with my labels on 10/10 pairs.


## A few observations

- BERT's NSP head was trained to distinguish "actual next sentence in the corpus" from "random sentence from elsewhere in the corpus", with 50/50 sampling. On my crafted pairs, "random" usually scores way below 0.5 and "follows" usually scores above. The model is doing roughly what I'd expect.
- Where it tends to be wrong: pairs where the topic matches but the logical follow doesn't (e.g., two unrelated cooking instructions about different recipes). NSP can get fooled by topic overlap because that's a major signal in pretraining.
- This is why later models (RoBERTa, modern BERT variants) dropped NSP — sentence order prediction (SOP) or just MLM alone gets you similar representations without the topic-matching shortcut.
- For real downstream use: I'd reach for a sentence embedding model (`sentence-transformers/all-MiniLM-L6-v2`) and cosine similarity, not the NSP head. The NSP head's value is mostly historical / pedagogical at this point.
